<a href="https://colab.research.google.com/github/Mustafa0291polat/AI-Arac-Kiralama-Platformu/blob/feature%2FKAN-12-ai-api-baglanti-testleri/Yaz%C4%B1l%C4%B1m_M%C3%BCh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

# .env dosyasındaki ortam değişkenlerini sisteme yükler
load_dotenv()

# API anahtarını ortam değişkenlerinden çeker
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("HATA: OPENAI_API_KEY bulunamadı. Lütfen .env dosyanızı kontrol edin.")

client = OpenAI(api_key=api_key)

def arama_motoru_filtre_cikar(kullanici_metni):
    """
    Kullanıcıdan gelen serbest arama metnini analiz ederek, veritabanı
    sorgusunda kullanılacak JSON filtre objesini döndürür.
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": """Sen "Yapay Zeka Destekli Bireyler Arası Araç Kiralama Platformu" için çalışan bir arama motoru asistanısın.
Görevin, kullanıcının girdiği serbest metni analiz ederek aşağıdaki formatta ve veri tiplerinde bir JSON objesi döndürmektir.
Başka hiçbir metin, açıklama veya markdown satırı yazma. Sadece geçerli bir JSON döndür.

Kurallar ve Varsayılan (Default) Değerler:
- eger kullanici bir kriteri hic belirtmemisse, parantez icinde yazan "Varsayilan" degerleri kullan.
- il: (Varsayılan: null) Türkiye'deki iller.
- ilce: (Varsayılan: null) Türkiye'deki ilçeler.
- min_fiyat: (Varsayılan: 0) Sayı olmalı.
- max_fiyat: (Varsayılan: 10000000) Sayı olmalı.
- min_yil: (Varsayılan: 1970) Sayı olmalı.
- max_yil: (Varsayılan: 2030) Sayı olmalı.
- yakit_tipi: (Varsayılan: "Tümü") "Benzinli", "Benzin & LPG", "Dizel", "Hibrit", "Elektrikli" veya "Tümü"
- vites_tipi: (Varsayılan: "Tümü") "Manuel", "Otomatik" veya "Tümü"
- min_km: (Varsayılan: 0) Sayı olmalı.
- max_km: (Varsayılan: 9999999) Sayı olmalı.
- kasa_tipi: (Varsayılan: "Tümü") "Cabrio", "Coupe", "Coupe 4 kapı", "Hatchback 3 kapı", "Hatchback 5 kapı", "Sedan", "Station Wagon", "MPV", "Roadster" veya "Tümü"
- motor_gucu: (Varsayılan: "Tümü") Kullanıcı beygir/HP gücü belirtirse aralığı yaz (Örn: "51 - 75 HP", "101 - 125 HP" vb.)
- motor_hacmi: (Varsayılan: "Tümü") Kullanıcı motor hacmi (cc veya litre) belirtirse uygun cm3 aralığını yaz (Örn: "1301 - 1600 cm3" vb.)
- cekis: (Varsayılan: "Tümü") "Önden Çekiş", "Arkadan İtiş", "4WD (Sürekli)", "AWD (Elektronik)" veya "Tümü"
- renk: (Varsayılan: "Tümü") Kullanıcı renk belirtirse yaz (Bej, Beyaz, Bordo vb.)
- garanti: (Varsayılan: "Tümü") "Evet", "Hayır" veya "Tümü"
- agir_hasar_kayitli: (Varsayılan: "Tümü") "Evet", "Hayır" veya "Tümü"
- kimden: (Varsayılan: "Tümü") "Sahibinden", "Kurumdan" veya "Tümü"
- marka: (Varsayılan: null) Örn: Renault, Toyota, BMW, Audi, Tofaş, BYD vb. Kullanıcı sadece model belirtse bile o modelin hangi markaya ait olduğunu bil ve bunu doldur (Örn: "Clio" derse marka "Renault" olur).
- model: (Varsayılan: null) Örn: Corolla, Megane, 3 Serisi, A3, Şahin, Dolphin vb.

Format (Tam olarak bu anahtarları kullan):
{
    "il": null, "ilce": null, "min_fiyat": 0, "max_fiyat": 10000000,
    "min_yil": 1970, "max_yil": 2030, "yakit_tipi": "Tümü", "vites_tipi": "Tümü",
    "min_km": 0, "max_km": 9999999, "kasa_tipi": "Tümü", "motor_gucu": "Tümü",
    "motor_hacmi": "Tümü", "cekis": "Tümü", "renk": "Tümü", "garanti": "Tümü",
    "agir_hasar_kayitli": "Tümü", "kimden": "Tümü", "marka": null, "model": null
}"""
                },
                {
                    "role": "user",
                    "content": kullanici_metni
                }
            ],
            max_tokens=1000,
            temperature=0.0
        )

        ham_yanit = response.choices[0].message.content.strip()

        # Markdown kalıntılarını temizleme
        if ham_yanit.startswith("```json"):
            ham_yanit = ham_yanit[7:-3]
        elif ham_yanit.startswith("```"):
            ham_yanit = ham_yanit[3:-3]

        return json.loads(ham_yanit)

    except (json.JSONDecodeError, Exception) as e:
        # Web sunucusunun çökmemesi için
        print(f"Sistem Hatası: {e}")
        return None

In [ ]:
# @title Test - Son Hali
import json
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('OPENAI'))

def filtreleme_uygun_arac(kullanici_metni):
    print("OpenAI API'sine istek gönderiliyor...\n")
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": """Sen "Yapay Zeka Destekli Bireyler Arası Araç Kiralama Platformu" için çalışan bir arama motoru asistanısın.
Görevin, kullanıcının girdiği serbest metni analiz ederek aşağıdaki formatta ve veri tiplerinde bir JSON objesi döndürmektir.
Başka hiçbir metin, açıklama veya markdown satırı yazma. Sadece geçerli bir JSON döndür.

Kurallar ve Varsayılan (Default) Değerler:
- eger kullanici bir kriteri hic belirtmemisse, parantez icinde yazan "Varsayilan" degerleri kullan.
- il: (Varsayılan: null) Türkiye'deki iller.
- ilce: (Varsayılan: null) Türkiye'deki ilçeler.
- min_fiyat: (Varsayılan: 0) Sayı olmalı.
- max_fiyat: (Varsayılan: 10000000) Sayı olmalı. Not: Maksimum fiyat sınırını araç fiyatları günümüzde yüksek olduğu için mantıklı bir üst limite çektim.
- min_yil: (Varsayılan: 1970) Sayı olmalı.
- max_yil: (Varsayılan: 2030) Sayı olmalı.
- yakit_tipi: (Varsayılan: "Tümü") "Benzinli", "Benzin & LPG", "Dizel", "Hibrit", "Elektrikli" veya "Tümü"
- vites_tipi: (Varsayılan: "Tümü") "Manuel", "Otomatik" veya "Tümü"
- min_km: (Varsayılan: 0) Sayı olmalı.
- max_km: (Varsayılan: 9999999) Sayı olmalı.
- kasa_tipi: (Varsayılan: "Tümü") "Cabrio", "Coupe", "Coupe 4 kapı", "Hatchback 3 kapı", "Hatchback 5 kapı", "Sedan", "Station Wagon", "MPV", "Roadster" veya "Tümü"
- motor_gucu: (Varsayılan: "Tümü") Kullanıcı beygir/HP gücü belirtirse aralığı yaz (Örn: "51 - 75 HP", "101 - 125 HP" vb.)
- motor_hacmi: (Varsayılan: "Tümü") Kullanıcı motor hacmi (cc veya litre) belirtirse uygun cm3 aralığını yaz (Örn: "1301 - 1600 cm3" vb.)
- cekis: (Varsayılan: "Tümü") "Önden Çekiş", "Arkadan İtiş", "4WD (Sürekli)", "AWD (Elektronik)" veya "Tümü"
- renk: (Varsayılan: "Tümü") Kullanıcı renk belirtirse yaz (Bej, Beyaz, Bordo vb.)
- garanti: (Varsayılan: "Tümü") "Evet", "Hayır" veya "Tümü"
- agir_hasar_kayitli: (Varsayılan: "Tümü") "Evet", "Hayır" veya "Tümü"
- kimden: (Varsayılan: "Tümü") "Sahibinden", "Kurumdan" veya "Tümü"
- marka: (Varsayılan: null) Örn: Renault, Toyota, BMW, Audi, Tofaş, BYD vb. Kullanıcı sadece model belirtse bile o modelin hangi markaya ait olduğunu bil ve bunu doldur (Örn: "Clio" derse marka "Renault" olur).
- model: (Varsayılan: null) Örn: Corolla, Megane, 3 Serisi, A3, Şahin, Dolphin vb.

Format (Tam olarak bu anahtarları kullan):
{
    "il": null, "ilce": null, "min_fiyat": 0, "max_fiyat": 10000000,
    "min_yil": 1970, "max_yil": 2030, "yakit_tipi": "Tümü", "vites_tipi": "Tümü",
    "min_km": 0, "max_km": 9999999, "kasa_tipi": "Tümü", "motor_gucu": "Tümü",
    "motor_hacmi": "Tümü", "cekis": "Tümü", "renk": "Tümü", "garanti": "Tümü",
    "agir_hasar_kayitli": "Tümü", "kimden": "Tümü", "marka": null, "model": null
}"""
                },
                {
                    "role": "user",
                    "content": kullanici_metni
                }
            ],
            max_tokens=1000,
            temperature=0.1 # Kesinlikle kurallara uyması ve halüsinasyon görmemesi için sıfıra çektik
        )

        ham_yanit = response.choices[0].message.content.strip()

        # JSON ayrıştırma (Eğer model başında/sonunda ```json gibi markdown dönerse onu temizler)
        if ham_yanit.startswith("```json"):
            ham_yanit = ham_yanit[7:-3]
        elif ham_yanit.startswith("```"):
            ham_yanit = ham_yanit[3:-3]

        islenmis_veri = json.loads(ham_yanit)

        print("✅ Başarılı! Yapay Zeka Tarafından Çıkarılan Filtreler (Veritabanı İçin):")
        for anahtar, deger in islenmis_veri.items():
            # Eğer değer null, "Tümü" veya varsayılan sayılar ise ekrana daha temiz basmak için
            # Sadece kullanıcının gerçekten değiştirdiği/etkilediği filtreleri görmek istersen burayı değiştirebilirsin.
            print(f"-> {anahtar}: {deger}")

        return islenmis_veri

    except json.JSONDecodeError as e:
        print("❌ Hata: API'den dönen yanıt geçerli bir JSON formatında değil.")
        print("Gelen yanıt:", ham_yanit)
        return None
    except Exception as e:
        print(f"❌ Beklenmeyen bir hata oluştu: {e}")
        return None


# ---------------- TEST SENARYOLARI ----------------

# Test 1: Sadece marka ve model (Diğer her şey varsayılan gelmeli)
print("TEST 1 BAŞLIYOR...")
test_1 = "Bana bir Egea lazım."
sonuc_1 = filtreleme_uygun_arac(test_1)
print("\n" + "="*50 + "\n")

# Test 2: Çok detaylı ve karmaşık bir metin
print("TEST 2 BAŞLIYOR...")
test_2 = "İstanbu Ataşehir'de, 2022 model ve üstü, en fazla 50K km'de, otomatik, kazasız (hasarsız), beyaz veya kırmızı renkli bmw arıyorum. fiyat farketmez."
sonuc_2 = filtreleme_uygun_arac(test_2)

TEST 1 BAŞLIYOR...
OpenAI API'sine istek gönderiliyor...

✅ Başarılı! Yapay Zeka Tarafından Çıkarılan Filtreler (Veritabanı İçin):
-> il: None
-> ilce: None
-> min_fiyat: 0
-> max_fiyat: 10000000
-> min_yil: 1970
-> max_yil: 2030
-> yakit_tipi: Tümü
-> vites_tipi: Tümü
-> min_km: 0
-> max_km: 9999999
-> kasa_tipi: Tümü
-> motor_gucu: Tümü
-> motor_hacmi: Tümü
-> cekis: Tümü
-> renk: Tümü
-> garanti: Tümü
-> agir_hasar_kayitli: Tümü
-> kimden: Tümü
-> marka: Fiat
-> model: Egea


TEST 2 BAŞLIYOR...
OpenAI API'sine istek gönderiliyor...

✅ Başarılı! Yapay Zeka Tarafından Çıkarılan Filtreler (Veritabanı İçin):
-> il: İstanbul
-> ilce: Ataşehir
-> min_fiyat: 0
-> max_fiyat: 10000000
-> min_yil: 2022
-> max_yil: 2030
-> yakit_tipi: Tümü
-> vites_tipi: Otomatik
-> min_km: 0
-> max_km: 50000
-> kasa_tipi: Tümü
-> motor_gucu: Tümü
-> motor_hacmi: Tümü
-> cekis: Tümü
-> renk: Beyaz
-> garanti: Tümü
-> agir_hasar_kayitli: Hayır
-> kimden: Tümü
-> marka: BMW
-> model: None
